In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer

/Users/snehprinceherenj/dev/Music-Discovery-Engine/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
for dirname, _, filenames in os.walk('./'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./CI-CD-DEPLOYMENT-PLAN.md
./requirements.txt
./songs_with_mood.parquet
./recommendation_engine.py
./notebook.ipynb
./README.md
./api.py
./embeddings.npy
./song_lyrics_preprocessed.parquet
./.venv/pyvenv.cfg
./.venv/.gitignore
./.venv/bin/jupyter-run
./.venv/bin/𝜋thon
./.venv/bin/Activate.ps1
./.venv/bin/python3
./.venv/bin/typer
./.venv/bin/pip3.14
./.venv/bin/python
./.venv/bin/debugpy
./.venv/bin/pip3
./.venv/bin/ipython
./.venv/bin/activate.fish
./.venv/bin/tiny-agents
./.venv/bin/isympy
./.venv/bin/torchrun
./.venv/bin/idna
./.venv/bin/f2py
./.venv/bin/ipython3
./.venv/bin/pip
./.venv/bin/httpx
./.venv/bin/transformers
./.venv/bin/tqdm
./.venv/bin/markdown-it
./.venv/bin/huggingface-cli
./.venv/bin/jupyter-troubleshoot
./.venv/bin/fastapi
./.venv/bin/pygmentize
./.venv/bin/hf
./.venv/bin/jupyter-migrate
./.venv/bin/torchfrtrace
./.venv/bin/uvicorn
./.venv/bin/activate
./.venv/bin/normalizer
./.venv/bin/numpy-config
./.venv/bin/jupyter-kernelspec
./.venv/bin/jupyter-kernel
./.venv/

In [3]:
n = 30_000

parquet_file = pq.ParquetFile('./song_lyrics_preprocessed.parquet')
df = next(parquet_file.iter_batches(batch_size=n)).to_pandas()
df = df[['id', 'title', 'artist', 'year', 'lyrics', 'language_cld3']]

In [4]:
df

,id,title,artist,year,lyrics,language_cld3
0,1,Killa Cam,Cam'ron,2004,"[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",en
1,3,Can I Live,JAY-Z,1996,"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",en
2,4,Forgive Me Father,Fabolous,2003,Maybe cause I'm eatin\nAnd these bastards fien...,en
3,5,Down and Out,Cam'ron,2004,[Produced by Kanye West and Brian Miller]\n\n[...,en
4,6,Fly In,Lil Wayne,2005,"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",en
...,...,...,...,...,...,...
29995,31694,Electric Kingdom vocal version,Twilight 22,1983,Electric kingdom\n\nDeep in the city people li...,en
29996,31695,The Corruptors Execution,E-40,1999,[Pimp C]\nHold up..\n\nIt's the motherfuckin C...,en
29997,31696,Do You Wana Freak,Freak Brothers,1997,Chorus:\n\nDo you wanna freak?\nDo you wanna f...,en
29998,31697,Family Affair HOF Fam,Reservoir Dogs,2009,"[Verse One] [Big Pooh]\nSince '07 I said, ""Fuc...",en


In [5]:
def build_song_text_corpus(row):
    parts = [
        str(row['title']),
        str(row['artist']),
        str(row['year']),
        str([row['lyrics']])[:500]
    ]

    return ' '.join([p for p in parts if p != 'nan'])

df['text_corpus'] = df.apply(build_song_text_corpus, axis=1)
df

,id,title,artist,year,lyrics,language_cld3,text_corpus
0,1,Killa Cam,Cam'ron,2004,"[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",en,Killa Cam Cam'ron 2004 ['[Chorus: Opera Steve ...
1,3,Can I Live,JAY-Z,1996,"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",en,Can I Live JAY-Z 1996 ['[Produced by Irv Gotti...
2,4,Forgive Me Father,Fabolous,2003,Maybe cause I'm eatin\nAnd these bastards fien...,en,Forgive Me Father Fabolous 2003 ['Maybe cause ...
3,5,Down and Out,Cam'ron,2004,[Produced by Kanye West and Brian Miller]\n\n[...,en,Down and Out Cam'ron 2004 ['[Produced by Kanye...
4,6,Fly In,Lil Wayne,2005,"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",en,Fly In Lil Wayne 2005 ['[Intro]\nSo they ask m...
...,...,...,...,...,...,...,...
29995,31694,Electric Kingdom vocal version,Twilight 22,1983,Electric kingdom\n\nDeep in the city people li...,en,Electric Kingdom vocal version Twilight 22 198...
29996,31695,The Corruptors Execution,E-40,1999,[Pimp C]\nHold up..\n\nIt's the motherfuckin C...,en,The Corruptors Execution E-40 1999 ['[Pimp C]\...
29997,31696,Do You Wana Freak,Freak Brothers,1997,Chorus:\n\nDo you wanna freak?\nDo you wanna f...,en,"Do You Wana Freak Freak Brothers 1997 [""Chorus..."
29998,31697,Family Affair HOF Fam,Reservoir Dogs,2009,"[Verse One] [Big Pooh]\nSince '07 I said, ""Fuc...",en,Family Affair HOF Fam Reservoir Dogs 2009 ['[V...


In [6]:
embeddings_path = './embeddings.npy'
model = SentenceTransformer('all-MiniLM-L6-v2')
if os.path.exists(embeddings_path):
    embeddings = np.load(embeddings_path)
    print(f"Loaded embeddings from {embeddings_path}")
else:
    embeddings = model.encode(
        df['text_corpus'].tolist(),
        batch_size=64,
        show_progress_bar=True
    )
    np.save('embeddings.npy', embeddings)
print(f"Embeddings shape: {embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12196.19it/s]


Loaded embeddings from ./embeddings.npy
Embeddings shape: (30000, 384)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

mood_tagged = './songs_with_mood.parquet'
if os.path.exists(mood_tagged):
    df = pd.read_parquet(mood_tagged)
else:
    def get_mood(lyrics):
        score = analyzer.polarity_scores(str(lyrics))['compound']
        if score > 0.3: return 'happy'
        elif score < -0.3: return 'sad'
        else: return 'neutral'

    df['mood'] = df['lyrics'].apply(get_mood)

    df.to_parquet('songs_with_mood.parquet', index=False)
df

,id,title,artist,year,lyrics,language_cld3,text_corpus,mood
0,1,Killa Cam,Cam'ron,2004,"[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",en,Killa Cam Cam'ron 2004 ['[Chorus: Opera Steve ...,sad
1,3,Can I Live,JAY-Z,1996,"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",en,Can I Live JAY-Z 1996 ['[Produced by Irv Gotti...,sad
2,4,Forgive Me Father,Fabolous,2003,Maybe cause I'm eatin\nAnd these bastards fien...,en,Forgive Me Father Fabolous 2003 ['Maybe cause ...,neutral
3,5,Down and Out,Cam'ron,2004,[Produced by Kanye West and Brian Miller]\n\n[...,en,Down and Out Cam'ron 2004 ['[Produced by Kanye...,sad
4,6,Fly In,Lil Wayne,2005,"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",en,Fly In Lil Wayne 2005 ['[Intro]\nSo they ask m...,sad
...,...,...,...,...,...,...,...,...
29995,31694,Electric Kingdom vocal version,Twilight 22,1983,Electric kingdom\n\nDeep in the city people li...,en,Electric Kingdom vocal version Twilight 22 198...,happy
29996,31695,The Corruptors Execution,E-40,1999,[Pimp C]\nHold up..\n\nIt's the motherfuckin C...,en,The Corruptors Execution E-40 1999 ['[Pimp C]\...,sad
29997,31696,Do You Wana Freak,Freak Brothers,1997,Chorus:\n\nDo you wanna freak?\nDo you wanna f...,en,"Do You Wana Freak Freak Brothers 1997 [""Chorus...",happy
29998,31697,Family Affair HOF Fam,Reservoir Dogs,2009,"[Verse One] [Big Pooh]\nSince '07 I said, ""Fuc...",en,Family Affair HOF Fam Reservoir Dogs 2009 ['[V...,sad


In [8]:
def infer_mood(query):
    score = analyzer.polarity_scores(query)['compound']
    if score > 0.3: return 'happy'
    elif score < -0.3: return 'sad'
    else: return 'neutral'

def recommend(query, top_k=7, filter_mood=True):
    query_vec = model.encode([query])
    scores = cosine_similarity(query_vec, embeddings)[0]

    results = df.copy()
    results['scores'] = scores

    if filter_mood:
        mood = infer_mood(query)
        if mood:
            results = results[results['mood'] == mood]

    return results.sort_values('scores', ascending=False).head(top_k)[['title', 'artist', 'year', 'mood', 'scores']]

results = recommend("rap energetic")
print(results)

                                   title                        artist  year  \
13834  E.P.G.H. East Points Greatest Hit                   Cool Breeze  1999   
15320           Hip-Hop Loud Rocks Remix          Dead Prez & Static-X  2000   
16563                Vicious Battle Raps                     DJ Format  2003   
4400                      Practice remix  Jack the Ripper and Exzactly  2011   
19355           Best Rapper in the World          Freestyle Fellowship  2001   
755                           God of Rap                        Afu-Ra  2005   
9970                            Get Down                    Biz Markie  2003   

        mood    scores  
13834  happy  0.592845  
15320  happy  0.543893  
16563  happy  0.536658  
4400   happy  0.531661  
19355  happy  0.529901  
755    happy  0.528992  
9970   happy  0.526159  


In [9]:
# Embedding statistics
print(f"Mean embedding value: {embeddings.mean():.8f}")
print(f"Embedding std dev: {embeddings.std():.4f}")

# Score distribution
print(f"Mean similarity score: {results['scores'].mean():.4f}")
print(f"Score range: [{results['scores'].min():.4f}, {results['scores'].max():.4f}]")

# Mood distribution
print("\nMood distribution:")
print(df['mood'].value_counts())

Mean embedding value: -0.00004606
Embedding std dev: 0.0510
Mean similarity score: 0.5414
Score range: [0.5262, 0.5928]

Mood distribution:
mood
sad        17710
happy      11770
neutral      520
Name: count, dtype: int64
